# D3-01 Generating prospective databases with premise

Audience:
- Participants who already completed the Brightway and `ecoinvent` steps from Days 1 and 2.

Prerequisites:
- `paris-lca-course-2026` exists and contains `ecoinvent-3.12-cutoff`.
- This notebook loads the repo-root `.env` file automatically and falls back to the embedded course IAM key if `IAM_FILES_KEY` is absent.

Learning goals:
- Build the course scenario list for REMIND-EU.
- Run `premise.NewDatabase(...)` on `ecoinvent 3.12 cutoff`.
- Write the generated scenario databases to Brightway.
- Export a datapackage for optional `trails` work.


## Outline

1. Confirm the input database and credentials.
2. Define the REMIND-EU scenarios used in the course.
3. Generate the updated scenario databases.
4. Write Brightway databases and a `trails` datapackage.
5. Check the exported artifacts.


In [1]:
import os
from pathlib import Path

import bw2data as bd
import pandas as pd
from dotenv import load_dotenv
from premise import NewDatabase, clear_cache

load_dotenv()

project_name = 'paris-lca-course-2026'
source_db = 'ecoinvent-3.12-cutoff'
system_model = 'cutoff'
source_version = '3.12'
export_name = 'paris-trails-datapackage'
default_iam_key = 'tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo='

bd.projects.set_current(project_name)

biosphere_name = next((name for name in bd.databases if 'biosphere' in name and "3.12" in name), None)
iam_key = os.environ.get('IAM_FILES_KEY', default_iam_key)

print('Project        :', project_name)
print('Source database:', source_db)
print('Biosphere      :', biosphere_name)
print('IAM key in env :', bool(os.environ.get('IAM_FILES_KEY')))
print('Active IAM key :', iam_key)


/opt/homebrew/Caskroom/miniforge/base/envs/lca-course/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


Project        : paris-lca-course-2026
Source database: ecoinvent-3.12-cutoff
Biosphere      : ecoinvent-3.12-biosphere
IAM key in env : True
Active IAM key : tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=


## Step 1 - Define the course scenarios

We compare two policy pathways across three years, always with `remind-eu` as the source of scenario data.


In [3]:
scenarios = [
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2025},
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2035},
    {'model': 'remind-eu', 'pathway': 'SSP2-NPi', 'year': 2050},
    # {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2025}, in principle, we do not need this one
    {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2035},
    {'model': 'remind-eu', 'pathway': 'SSP2-PkBudg1000', 'year': 2050},
]

scenario_table = pd.DataFrame(scenarios)
scenario_table


,model,pathway,year
0,remind-eu,SSP2-NPi,2025
1,remind-eu,SSP2-NPi,2035
2,remind-eu,SSP2-NPi,2050
3,remind-eu,SSP2-PkBudg1000,2035
4,remind-eu,SSP2-PkBudg1000,2050


## Step 2 - Generate the prospective database object

This step can take a while. Keep the notebook attached to the same `lca-course` kernel while it runs.  
The notebook loads `.env` from the repository root first, and only falls back to the embedded course IAM key if `IAM_FILES_KEY` is absent.


In [5]:
if source_db not in bd.databases:
    raise ValueError(f'Missing source database: {source_db}')

ndb = NewDatabase(
    scenarios=scenarios,
    source_db=source_db,
    source_version=source_version,
    key=iam_key.encode(),
    system_model=system_model,
    biosphere_name=biosphere_name,
)

ndb.update()
print('Scenario update complete.')


Cannot find the IAM scenario file at /Users/romain/GitHub/premise/premise/data/iam_output_files/remind-eu_SSP2-NPi. Will check online.
Cannot find the IAM scenario file at /Users/romain/GitHub/premise/premise/data/iam_output_files/remind-eu_SSP2-NPi. Will check online.
Cannot find the IAM scenario file at /Users/romain/GitHub/premise/premise/data/iam_output_files/remind-eu_SSP2-NPi. Will check online.
Cannot find the IAM scenario file at /Users/romain/GitHub/premise/premise/data/iam_output_files/remind-eu_SSP2-PkBudg1000. Will check online.
Cannot find the IAM scenario file at /Users/romain/GitHub/premise/premise/data/iam_output_files/remind-eu_SSP2-PkBudg1000. Will check online.
premise v.(2, 3, 9)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing ne

100%|█████████████████████████████████| 26533/26533 [00:00<00:00, 412637.35it/s]


Adding exchange data to activities


100%|████████████████████████████████| 892943/892943 [00:44<00:00, 19961.38it/s]


Filling out exchange data


...

remind-eu_SSP2-NPi.csv downloaded successfully.
Reading remind-eu_SSP2-NPi as CSV file
Found file: remind-eu_SSP2-NPi
Reading remind-eu_SSP2-NPi as CSV file
Found file: remind-eu_SSP2-NPi
Reading remind-eu_SSP2-NPi as CSV file
remind-eu_SSP2-PkBudg1000.csv not found locally. Downloading...


remind-eu_SSP2-PkBudg1000.csv: 23.2MB [00:08, 2.65MB/s]


remind-eu_SSP2-PkBudg1000.csv downloaded successfully.
Reading remind-eu_SSP2-PkBudg1000 as CSV file
Found file: remind-eu_SSP2-PkBudg1000
Reading remind-eu_SSP2-PkBudg1000 as CSV file
Done!


Processing scenarios for all sectors: 100%|█| 5/5 [09:56<00:00, 119.29

Done!

Scenario update complete.


## Step 3 - Write Brightway databases and a datapackage

We use deterministic database names so that later Brightway and Activity Browser comparisons stay readable.


In [ ]:
ndb.write_db_to_brightway()

Write new database(s) to Brightway.
Running all checks...
Minor anomalies found: check the change report.
16:10:06+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|████████████████████████████████████| 48221/48221 [01:03<00:00, 758.43it/s]

16:11:11+0200 [info     ] Vacuuming database            


Created database: ei_cutoff_3.12_remind-eu_SSP2-NPi_2025 2026-04-12
Running all checks...
Minor anomalies found: check the change report.
16:13:03+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|████████████████████████████████████| 48224/48224 [01:04<00:00, 747.80it/s]

16:14:10+0200 [info     ] Vacuuming database            


## Step 4 - Check the exported artifacts

The `premise` datapackage is the bridge to optional `trails` work on Day 4.


In [ ]:
export_path = Path.cwd() / 'export' / 'datapackage' / f'{export_name}.zip'

print('Datapackage path:', export_path)
print('Exists          :', export_path.exists())
print('Generated DBs   :', [name for name in db_names if name in bd.databases])
